In [ ]:
import json
import sys
import numpy as np
from collections import Counter

from pantry.pipeline import PantryPipeline

# Load pipeline
print("Loading pipeline...")
pipeline = PantryPipeline()
print("Pipeline ready!")

# Load test prompts
with open("evaluation/test_prompts.json") as f:
    test_prompts = json.load(f)
print(f"Loaded {len(test_prompts)} test prompts")


In [ ]:
# Run all prompts through the parser
results = []
for tp in test_prompts:
    parsed = pipeline.parse(tp["prompt"])
    results.append({
        "prompt": tp["prompt"],
        "category": tp["category"],
        "expected_ingredients": set(tp["expected_ingredients"]),
        "extracted_ingredients": set(parsed["ingredients"]),
        "expected_constraints": tp["expected_constraints"],
        "extracted_constraints": parsed["constraints"],
        "expected_mood": set(tp["expected_mood_tokens"]),
        "extracted_mood": set(parsed["mood_tokens"]),
    })
print(f"Processed {len(results)} prompts")


## Ingredient Extraction: Precision & Recall

In [ ]:
total_tp = 0  # true positives
total_fp = 0  # false positives
total_fn = 0  # false negatives

for r in results:
    expected = r["expected_ingredients"]
    extracted = r["extracted_ingredients"]
    tp = len(expected & extracted)
    fp = len(extracted - expected)
    fn = len(expected - extracted)
    total_tp += tp
    total_fp += fp
    total_fn += fn

precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Ingredient Extraction Metrics:")
print(f"  Precision: {precision:.2%}")
print(f"  Recall:    {recall:.2%}")
print(f"  F1 Score:  {f1:.2%}")
print(f"  (TP={total_tp}, FP={total_fp}, FN={total_fn})")


## Constraint Extraction: Accuracy

In [ ]:
time_correct = 0
effort_correct = 0
total = len(results)

for r in results:
    if r["extracted_constraints"]["time"] == r["expected_constraints"]["time"]:
        time_correct += 1
    if r["extracted_constraints"]["effort"] == r["expected_constraints"]["effort"]:
        effort_correct += 1

print(f"Constraint Extraction Accuracy:")
print(f"  Time accuracy:   {time_correct}/{total} = {time_correct/total:.2%}")
print(f"  Effort accuracy: {effort_correct}/{total} = {effort_correct/total:.2%}")
print(f"  Overall:         {(time_correct + effort_correct)/(2*total):.2%}")


## Mood Token Detection Rate

In [ ]:
mood_expected_total = 0
mood_detected = 0

for r in results:
    expected = r["expected_mood"]
    extracted = r["extracted_mood"]
    mood_expected_total += len(expected)
    mood_detected += len(expected & extracted)

detection_rate = mood_detected / mood_expected_total if mood_expected_total > 0 else 0
print(f"Mood Token Detection:")
print(f"  Expected tokens:  {mood_expected_total}")
print(f"  Detected tokens:  {mood_detected}")
print(f"  Detection rate:   {detection_rate:.2%}")


## Partial Input Handling (Pass/Fail)

In [ ]:
errors = []
for tp in test_prompts:
    try:
        result = pipeline.search(tp["prompt"])
        # Just needs to not crash - results can be empty
    except Exception as e:
        errors.append({"prompt": tp["prompt"], "error": str(e)})

if errors:
    print(f"FAIL: {len(errors)} prompts caused errors:")
    for e in errors:
        print(f"  - "{e['prompt']}": {e['error']}")
else:
    print(f"PASS: All {len(test_prompts)} prompts handled without errors")


## Summary Metrics Table

In [ ]:
print("=" * 60)
print(f"{'Metric':<35} {'Value':>15}")
print("=" * 60)
print(f"{'Ingredient Precision':<35} {precision:>14.1%}")
print(f"{'Ingredient Recall':<35} {recall:>14.1%}")
print(f"{'Ingredient F1':<35} {f1:>14.1%}")
print(f"{'Constraint Time Accuracy':<35} {time_correct/total:>14.1%}")
print(f"{'Constraint Effort Accuracy':<35} {effort_correct/total:>14.1%}")
print(f"{'Mood Detection Rate':<35} {detection_rate:>14.1%}")
print(f"{'Partial Input Handling':<35} {'PASS' if not errors else 'FAIL':>15}")
print("=" * 60)


## Example: Pipeline Working Well

In [ ]:
# Show 5 examples where the pipeline works well
good_examples = []
for r in results:
    expected_ing = r["expected_ingredients"]
    extracted_ing = r["extracted_ingredients"]
    ing_match = expected_ing == extracted_ing or (not expected_ing and not extracted_ing)
    
    time_match = r["extracted_constraints"]["time"] == r["expected_constraints"]["time"]
    effort_match = r["extracted_constraints"]["effort"] == r["expected_constraints"]["effort"]
    
    mood_match = r["expected_mood"] <= r["extracted_mood"]  # all expected found
    
    if ing_match and time_match and effort_match and mood_match:
        good_examples.append(r)

print(f"Perfect matches: {len(good_examples)}/{len(results)}")
print()
for ex in good_examples[:5]:
    print(f"Prompt: "{ex['prompt']}"")
    print(f"  Category: {ex['category']}")
    if ex["extracted_ingredients"]:
        print(f"  Ingredients: {sorted(ex['extracted_ingredients'])}")
    if ex["extracted_constraints"]["time"] or ex["extracted_constraints"]["effort"]:
        print(f"  Constraints: {ex['extracted_constraints']}")
    if ex["extracted_mood"]:
        print(f"  Mood: {sorted(ex['extracted_mood'])}")
    print()


## Failure Case Analysis

In [ ]:
# Show cases where extraction didn't match expectations
failures = []
for r in results:
    issues = []
    
    missing_ing = r["expected_ingredients"] - r["extracted_ingredients"]
    extra_ing = r["extracted_ingredients"] - r["expected_ingredients"]
    if missing_ing:
        issues.append(f"Missing ingredients: {sorted(missing_ing)}")
    if extra_ing:
        issues.append(f"Extra ingredients: {sorted(extra_ing)}")
    
    if r["extracted_constraints"]["time"] != r["expected_constraints"]["time"]:
        issues.append(f"Time: expected {r['expected_constraints']['time']}, got {r['extracted_constraints']['time']}")
    if r["extracted_constraints"]["effort"] != r["expected_constraints"]["effort"]:
        issues.append(f"Effort: expected {r['expected_constraints']['effort']}, got {r['extracted_constraints']['effort']}")
    
    missing_mood = r["expected_mood"] - r["extracted_mood"]
    if missing_mood:
        issues.append(f"Missing mood tokens: {sorted(missing_mood)}")
    
    if issues:
        failures.append({"prompt": r["prompt"], "category": r["category"], "issues": issues})

print(f"Prompts with issues: {len(failures)}/{len(results)}")
print()
for f in failures[:5]:
    print(f"Prompt: "{f['prompt']}"")
    print(f"  Category: {f['category']}")
    for issue in f["issues"]:
        print(f"  - {issue}")
    print()

# Analysis
if failures:
    print("
--- Failure Pattern Analysis ---")
    issue_types = Counter()
    for f in failures:
        for issue in f["issues"]:
            if "Missing ingredients" in issue:
                issue_types["Missing ingredients"] += 1
            elif "Extra ingredients" in issue:
                issue_types["Extra ingredients"] += 1
            elif "Time" in issue:
                issue_types["Time mismatch"] += 1
            elif "Effort" in issue:
                issue_types["Effort mismatch"] += 1
            elif "Missing mood" in issue:
                issue_types["Missing mood"] += 1
    for itype, count in issue_types.most_common():
        print(f"  {itype}: {count} cases")


## Per-Category Breakdown

In [ ]:
categories = {}
for r in results:
    cat = r["category"]
    if cat not in categories:
        categories[cat] = {"total": 0, "perfect": 0}
    categories[cat]["total"] += 1
    
    expected_ing = r["expected_ingredients"]
    extracted_ing = r["extracted_ingredients"]
    ing_ok = expected_ing == extracted_ing or (not expected_ing and not extracted_ing)
    time_ok = r["extracted_constraints"]["time"] == r["expected_constraints"]["time"]
    effort_ok = r["extracted_constraints"]["effort"] == r["expected_constraints"]["effort"]
    mood_ok = r["expected_mood"] <= r["extracted_mood"]
    
    if ing_ok and time_ok and effort_ok and mood_ok:
        categories[cat]["perfect"] += 1

print(f"{'Category':<20} {'Perfect':>8} {'Total':>8} {'Rate':>8}")
print("-" * 46)
for cat in ["ingredient-only", "mood-only", "constraint-only", "mixed", "edge-case"]:
    if cat in categories:
        c = categories[cat]
        print(f"{cat:<20} {c['perfect']:>8} {c['total']:>8} {c['perfect']/c['total']:>7.0%}")
